# DFX4ML System Test Notebook

This notebook demonstrates a full end-to-end test of the **DFX4ML** (Dynamic Function eXchange for Machine Learning) system running on a PYNQ-based FPGA board.

## What This Notebook Does

The system uses **partial reconfiguration** to dynamically swap hardware accelerator kernels (Reconfigurable Modules, RMs) at runtime without fully reprogramming the FPGA. The test pipeline:

1. Load the static full bitstream (base overlay)
2. Initialize the DFX Manager (sequencer) and DFX Controller
3. Load two partial bitstreams (`rm_0`, `rm_1`) into CMA memory
4. Feed input data through the pipeline via DMA
5. The DFX Manager orchestrates: load → compute → store → reconfigure → repeat
6. Read back results and verify correctness

> **Hardware Setup Required:** This notebook must run on a PYNQ board with the correct bitstreams placed in `hw/`.

In [ ]:
# import the library
## pynq
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
## standard python
import asyncio
import os
import time
import numpy as np
## our driver
import driver.cap           as cap
import driver.mem_alloc     as dataAlloc  # import the memory allocation library
import driver.dfx_unified   as dfx_unified
import driver.dfx_mgs_debug as dfx_mgs_debug


## Step 1 — Project Configuration

Set file paths, bitstream names, and data shapes used throughout the notebook.

| Variable | Description |
|---|---|
| `FULL_BS_NAME` | Static (full) bitstream — programs the entire PL |
| `PAR_BS_NAME_0/1` | Partial bitstreams for RM slot 0 and 1 |
| `AMT_QUERY` | Number of input samples to process |
| `AMT_SLOT` | Number of reconfigurable slots in the DFX system |
| `input_x` | Synthetic integer input data saved to disk as `.npy` |

In [ ]:
# path parameter
PRJ_DIR         = os.getcwd()
PRJ_HW_DIR      = PRJ_DIR + '/hw/'
PRJ_TC_DIR      = PRJ_DIR + '/data/'
DFX_CONFIG_FILE = 'dfx_ctrl_con.txt'
# reconfigurable module paramter
AMT_SLOT        = 2
FULL_BS_NAME    = 'system.bin'
PAR_BS_NAME_0   = 'rm_0.bin'  ###### dma to magic stream 1
PAR_BS_NAME_1   = 'rm_1.bin'  ###### magic stream 1 to magic stream 2
# input data parameter
INPUT_DATA_NAME = "input_x.npy"
AMT_QUERY       = 100
INPUT_SHAPE     = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
OUTPUT_SHAPE    = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes

In [ ]:
input_x = (np.arange(AMT_QUERY, dtype=np.int32)+48).reshape(INPUT_SHAPE)
np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)

## Step 2 — Load the Full Bitstream (Static Overlay)

Before loading the overlay, switch the PL configuration interface to **PCAP** (Processor Configuration Access Port), which is required for loading the full bitstream from the PS side.

Then load the full bitstream via PYNQ's `Overlay`. This programs the entire PL and exposes all IP blocks as Python objects.

In [ ]:
cap.change_pl_config_mode("pcap", True, "")

In [ ]:
# load the overlay
overlay = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

## Step 3 — Set Up Interrupt

Register the interrupt pin from `dfx_unified_0` so the software can **asynchronously wait** for the hardware to signal completion instead of busy-polling.

The interrupt fires when the DFX Manager finishes processing all slots.

In [ ]:
#### create the interrupt pin
overlay.interrupt_pins

In [ ]:
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')  # index 0 from your mapping

## Step 4 — Access IP Blocks

Get handles to the three main hardware IPs inside the `dfx_unified_0` block:

| Python Handle | Hardware IP | Role |
|---------------|---|---|
| `dma_ip`      | AXI DMA | Moves data between DDR and the reconfigurable region |
| `dfx_ctrl`    | DFX Controller | Triggers partial reconfiguration via ICAP |
| `dfx_mng`     | DFX Manager | Orchestrates the full DMA→compute→reconfig sequence |

In [ ]:
dfx_unifed_ip = overlay.dfx_unified_0

In [ ]:
#### get the device
dma_ip   = dfx_unifed_ip.dfx_dma
dfx_ctrl = dfx_unifed_ip.dfx_ctrl
dfx_mng  = dfx_unifed_ip.dfx_mng
pr_ctrl  = dfx_unifed_ip.pr_ctrl

## Step 5 — Configure the DFX Controller

Load the DFX Controller metadata from `dfx_ctrl_con.txt`. This file maps each **slot index** to its register space in the ICAP address map.

After config, switch the PL interface from **PCAP → ICAP** so that partial bitstreams can be loaded directly from the PL fabric (required for runtime partial reconfiguration).

In [ ]:
#### configure the dfx controller ip to match the address space
dfx_ctrl.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfx_ctrl.BLS_REGID)

In [ ]:
### change reconfigure mode
cap.change_pl_config_mode("icap", True, "")

## Step 6 — Initial System Reset

Shut down both the DFX Manager and DFX Controller engines to ensure they are in a clean idle state before configuring the data pipeline.

> This is a **soft reset** — it clears internal state machines without reprogramming the PL.

In [ ]:
dfx_ctrl.print_status()

In [ ]:
# shutdown all system
dfx_mng .shutdown_engine()
dfx_ctrl.shutdown_engine()

## Step 7 — Initialize the DFX Manager (Sequencer)

The **DFX Manager** is the central orchestrator. It holds a **slot table** where each slot describes one phase of the pipeline:

```
Slot 0: DMA load input  →  trigger RM_0  →  (no store)
Slot 1: (no load)       →  trigger RM_1  →  DMA store output
```

Each slot entry contains:
- `srcPhyAddr / srcSz` — source buffer for DMA read (input data)
- `dstPhyAddr / dstSz` — destination buffer for DMA write (output data)
- `loadMask / storeMask` — which DMA channels to activate
- `intrMask` — whether to fire an interrupt after this slot

The manager also needs the **physical addresses** of the DMA and DFX Controller so it can issue commands directly over AXI.

In [ ]:
# get physical address of dma and dfx controller
dmaPhyAddr     =  0xA003_0000 #overlay.ip_dict['dataMovement/axi_dma_0']['phys_addr']
prPhyAddr      =  0xA005_0000
dfxCtrlPhyAddr =  0 #overlay.ip_dict['PRcontroller/dfx_controller_0']['phys_addr']

print("dma physical address: "      , hex(dmaPhyAddr)    )
print("dfx  Ctrl physical address: ", hex(dfxCtrlPhyAddr))

In [ ]:
##### initialize magic seq
print("------ before init magic seq------")
print(dfx_mng.print_debug())

print("------ init magic sequence METADATA bank 0 -------------------------")
dfx_mng.set_end_cnt(AMT_SLOT - 1) ### use the last index
dfx_mng.set_dma_addr(dmaPhyAddr)
dfx_mng.set_dfx_addr(dfxCtrlPhyAddr)
dfx_mng.set_pr_ctrl_addr(prPhyAddr)
dfx_mng.set_batch_size(AMT_QUERY)
dfx_mng.set_intr_ena(1)
dfx_mng.set_intr(1)  # woc  command 1 to set the interrupt to 0
dfx_mng.set_round_trip(0)  # set round trip to 0, no need to wait for the dma to finish
inputX = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
if(inputX.shape != INPUT_SHAPE):
    raise Exception(f"inputX shape is {inputX} expect {INPUT_SHAPE}")

#inputX = np.random.rand(*INPUT_SHAPE).astype(np.float32)
print("-------------init all data buffer -------------")
buf_input   , buf_input_phya   , buf_input_sz    = dataAlloc.alloc_data_uint(alloc_shape= INPUT_SHAPE , alloc_type=np.int32, input_x = inputX)
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.alloc_data_uint(alloc_shape= OUTPUT_SHAPE, alloc_type=np.int32)
buf_input.flush()
print("------------- init dfx mng ------------------")
#                         [srcPhyAddr    ,        srcSz,  dstPhyAddr,      dstSz,st,pr,loadMask, storeMask, intrMask, profile_exec]
dfx_mng.set_whole_slot(0, [buf_input_phya, buf_input_sz,           0,          0, 0, 0,  0b0001, 0b0010, 0, 0])
dfx_mng.set_whole_slot(1, [             0,            0, buf_out_phy, buf_out_sz, 0, 0,  0b0010, 0b0001, 0, 0])

print("------------- after init magic seq------")
print(dfx_mng.print_debug())

In [ ]:
print("------------- pr controller status------")
dfx_unifed_ip.dfx_man.grant_decoupler_to_ps()
dfx_unifed_ip.dfx_man.release_decup()
dfx_unifed_ip.pr_ctrl.print_status()

## Step 8 — Load Partial Bitstreams into CMA Memory

Allocate **Contiguous Memory Allocation (CMA)** buffers for each partial bitstream file and copy the bitstream data into them.

The DFX Controller needs the bitstreams in physically contiguous memory so it can DMA them directly to ICAP for reconfiguration.

- `rm_0.bin` → partial bitstream for Reconfigurable Module 0
- `rm_1.bin` → partial bitstream for Reconfigurable Module 1

Then register the CMA addresses and sizes as metadata entries in the DFX Controller so it knows where each partial bitstream lives.

In [ ]:
##### initialize dfx controller
print("------ allocate bit steram CMA for each trigger ------")

######## set trigger 0
d0_ip_buf, d0_addr, d0_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_0)
######## set trigger 1
d1_ip_buf, d1_addr, d1_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_1)

In [ ]:
##### initialize dfx controller2
dfx_ctrl.set_simple_meta_data(0, d0_addr, d0_size)
dfx_ctrl.set_simple_meta_data(1, d1_addr, d1_size)

In [ ]:
##### check dfx controller3
dfx_ctrl.print_status()
dfx_ctrl.print_simple_meta_data(0)
dfx_ctrl.print_simple_meta_data(1)

## Step 9 — Initial DFX Trigger (Load First RM)

Before starting the DFX Manager engine, manually trigger the DFX Controller to load **RM_0** (slot 0) so the reconfigurable region has a valid bitstream from the start.

`dfx_ctrl.restart_no_status()` restarts the controller state machine without waiting for a status handshake.

In [ ]:
dfx_unifed_ip.dfx_man.grant_decoupler_to_dfx_ctrl()
dfx_ctrl.trig(0)
dfx_ctrl.restart_no_status()

In [ ]:
dfx_ctrl.print_status()

## Step 10 — Execute Pipeline and Wait for Interrupt

Start the DFX Manager engine and **asynchronously wait** for the hardware interrupt that signals all slots have been processed.

The hardware pipeline runs autonomously:
```
[DDR] --DMA--> [RM_0] --reconfig--> [RM_1] --DMA--> [DDR]
```

`time.perf_counter()` measures the wall-clock time from engine start to interrupt receipt.

In [ ]:
##### start dfx controller3
async def startExecAndWait4Intr():
    start_time = time.perf_counter()  # Start timing
    dfx_mng.clear_intr()
    dfx_mng.start_engine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        print("interrupt")
        print(f"Elapsed time: {end_time - start_time:.6f} seconds")
        break

In [ ]:
loop2 = asyncio.get_event_loop()

In [ ]:
task2 = loop2.create_task(startExecAndWait4Intr())
loop2.run_until_complete(task2)

In [ ]:
print(dfx_mng.print_debug())

In [ ]:
dfx_mng.shutdown_engine()

In [ ]:
print(dfx_mng.print_debug())

## Step 11 — Read Back and Verify Results

Invalidate the output cache buffer so the CPU sees the latest data written by the DMA engine, then read it back as a NumPy array.

**Expected result:** Since both RMs in this test are identity pass-throughs, `output == input_x` should be `True`.

In [ ]:
buf_out.invalidate()
np_parRes = np.array(buf_out, dtype=np.int32)
print(np_parRes)

In [ ]:
print(f"Compare input_x and np_parRes: {np.array_equal(input_x, np_parRes)}")

## Step 12 — Debug: Read DFX Manager State

Read internal debug registers from the MGS (Magic Sequence) block via GPIO to inspect the final hardware state.

Useful for diagnosing sequencer issues if the result comparison above fails.

In [ ]:
mgs_dbg = dfx_mgs_debug.DFX_mgs_debug(overlay.axi_gpio_0)
mgs_dbg.read_mgs_meta()